# Fingerprints to SMILES amb MolForge
Transforma una taula amb **fingerprints** en **SMILES** fent servir MolForge.

**Entrada**: CSV amb columna `fingerprints_input_ECPF4`.

**Sortida**: CSV amb columnes `fingerprints_input_ECFP4` i `SMILES_output_ECPF4`.

## Imports

In [1]:
# Per definir l'arrel del projecte
import os

# Pandas pels dataframes
import pandas as pd
import numpy as np # pels NaN

# Per cridar MolForge i guardar-ne l'output
from MolForge import main as molforge_main
from MolForge import predict as mf
import sys
import io
from contextlib import redirect_stdout

## Inputs (part a editar)

Arrel del projecte

In [2]:
#os.chdir("/export/home/ddiestre/MolForge_Testing")
os.chdir("/mnt/c/Users/david/Desktop/Uni/MolForge_Testing")
#os.chdir("/mnt/d/MolForge_Testing")

Paràmetres de MolForge

In [3]:
FP_NAME = "ECFP4"
MODEL_TYPE = "smiles"  # ["smiles", "selfies"]
DECODE = "greedy"  # ["greedy", "beam"]
CHECKPOINT_NAME = "ECFP4_smiles_checkpoint.pth"

Fitxer de fingerprints preprocessat (path a partir de MolForge_Testing/)

In [4]:
input_path = "data/MolForge_input/MolForge_MFinput_2000_ECFP4_noise0_0_6_noise1_0_6.csv"
#input_path = "data/MolForge_input/CoCoGraph_MFinput_2000_novel.csv"
#input_path = "data/MolForge_input/CoCoGraph_MFinput_2000_lt70atoms.csv"

in_col_name = "fingerprints_input_" + FP_NAME

Fitxer en que guardar l'output (path a partir de MolForge_Testing/)

In [5]:
output_path = "data/MolForge_output/MolForge_MFoutput_2000_ECFP4_noise0_0_6_noise1_0_6.csv"
#output_path = "data/MolForge_output/CoCoGraph_MFoutput_2000_novel.csv"
#output_path = "data/MolForge_output/CoCoGraph_MFoutput_2000_lt70atoms.csv"

out_col_name = "SMILES_output_" + FP_NAME

## 1. Lectura del fitxer

In [6]:
# Lectura del fitxer
df = pd.read_csv(input_path, sep = ',', index_col = 0)
df.head(5)

,SMILES_input,fingerprints_input_ECFP4
id,,
1,C C 1 = C ( C = C C ( = C 1 ) C ( C ) ( C ) C ...,1 13 33 80 94 114 118 227 283 351 392 404 422 ...
2,C ( C ( F ) ( F ) F ) ( C ( F ) ( F ) [18F] ) F,1 114 136 218 322 456 545 561 676 1050 1059 10...
3,C C 1 = C C = C C = C 1 C ( C ( = O ) N C 2 C ...,14 80 106 145 196 237 241 325 330 383 387 391 ...
4,C C O C ( = O ) C 1 = C C = C ( C = C 1 ) N C ...,5 41 63 80 89 121 129 145 147 162 191 202 216 ...
5,C 1 [C@H] ( C N ( C 1 = O ) C C 2 = C C = C C ...,80 93 101 172 197 221 231 295 311 314 366 378 ...


## 2. Execució de Molforge

In [7]:
# Estat global per guardar model i args després de la primera inferència
_MF_STATE = {
    "model": None,
    "args": None,
}


def carregar_model_i_args_un_cop(fp_name, model_type, checkpoint, decode):
    """
    Fa una crida a molforge_main() amb una fingerprint dummy, però
    amb un 'hook' sobre mf.inference per capturar el `model` i `args`
    que MolForge utilitza internament. El model i els args es carreguen
    una sola vegada i es guarden a _MF_STATE.
    """

    # Si ja l'hem carregat abans en aquesta sessió de notebook, el reutilitzem
    if _MF_STATE["model"] is not None and _MF_STATE["args"] is not None:
        return _MF_STATE["model"], _MF_STATE["args"]

    # 1) Guardem la funció d'inferència original
    original_inference = mf.inference

    def hook_inference(*f_args, **f_kwargs):
        """
        Hook que es cridarà en lloc d'mf.inference la PRIMERA vegada.
        Guarda model i args a _MF_STATE i després delega a l'original.
        La signatura exacta no importa perquè fem servir *args, **kwargs.
        """
        # Per com està escrit predict.py, la signatura és:
        # inference(model, input_sentence, method, args, return_attn=False)
        # → model = f_args[0], args = f_args[3]
        model = f_args[0]
        args = f_args[3]

        if _MF_STATE["model"] is None:
            _MF_STATE["model"] = model
            _MF_STATE["args"] = args

        # Fem la inferència normalment (una sola vegada, amb input dummy)
        return original_inference(*f_args, **f_kwargs)

    # 2) Substituïm temporalment mf.inference pel hook
    mf.inference = hook_inference

    # 3) Preparem sys.argv per a una única crida "dummy"
    original_argv = sys.argv
    sys.argv = [
        "",
        f"--fp={fp_name}",
        f"--model_type={model_type}",
        "--input=1",                # input dummy, només per activar main()
        f"--checkpoint={checkpoint}",
        f"--decode={decode}",
    ]

    # 4) Cridem molforge_main amb la sortida redirigida (logs → silenci)
    buf = io.StringIO()
    try:
        with redirect_stdout(buf):
            molforge_main()
    finally:
        # Restaurem sys.argv i la funció d'inferència original
        sys.argv = original_argv
        mf.inference = original_inference

    # 5) Comprovem que hem capturat model i args
    if _MF_STATE["model"] is None or _MF_STATE["args"] is None:
        raise RuntimeError(
            "No s'ha pogut capturar el model i/o els args des de MolForge. "
            "Revisa si MolForge.predict.inference s'està cridant dins de main()."
        )

    return _MF_STATE["model"], _MF_STATE["args"]

In [8]:
def run_molforge_batch_wrapper(
    df,
    fp_col,
    fp_name,
    model_type,
    checkpoint_name,
    decode,
):
    """
    Utilitza MolForge amb el model carregat UNA sola vegada.
    - Carrega model+args amb `carregar_model_i_args_un_cop(...)`
    - Fa un bucle per totes les fingerprints vàlides
    - Crida directament `mf.inference(model, fp_str, args.decode, args, ...)`
      capturant el stdout per extreure "Result: ..."
    - Afegeix una columna `SMILES_out` al DataFrame.
    """

    df_out = df.copy()
    df_out["SMILES_out"] = np.nan

    # 1) Files vàlides (descartem NaN i "InvalidSMILE")
    mask_valid = (~df_out[fp_col].isna()) & (df_out[fp_col] != "InvalidSMILE")
    idx_valid = df_out.index[mask_valid]
    n_valid = len(idx_valid)

    if n_valid == 0:
        print("No hi ha fingerprints vàlides per passar a MolForge.")
        return df_out

    print(
        f"Passant MolForge per {n_valid} fingerprints vàlides "
        f"(model carregat una sola vegada)..."
    )

    # 2) Carreguem model i args UNA sola vegada (i es cachegen a _MF_STATE)
    model, args = carregar_model_i_args_un_cop(
        fp_name=fp_name,
        model_type=model_type,
        checkpoint=checkpoint_name,
        decode=decode,
    )

    # 3) Bucle sobre les fingerprints vàlides
    n_done = 0
    for idx in idx_valid:
        fp_str = str(df_out.at[idx, fp_col]).strip()

        # Capturem stdout de la inferència per aquesta fingerprint
        buf = io.StringIO()
        with redirect_stdout(buf):
            # IMPORTANT: fem servir la funció d'inferència original del mòdul
            mf.inference(model, fp_str, args.decode, args)

        out = buf.getvalue()

        # Busquem la línia "Result: ..."
        pred_smi = np.nan
        for line in out.splitlines():
            line = line.strip()
            if line.startswith("Result:"):
                pred_smi = line.split("Result:", 1)[1].strip().replace(" ", "")
                break

        df_out.at[idx, "SMILES_out"] = pred_smi

        n_done += 1
        print(f"\r[{n_done}/{n_valid}]", end="", flush=True)

    print()  # salt de línia final
    return df_out

In [9]:
df = run_molforge_batch_wrapper(
    df=df,
    fp_col=in_col_name,      # p. ex. "fingerprints_input_ECFP4"
    fp_name=FP_NAME,         # "ECFP4"
    model_type=MODEL_TYPE,   # "smiles"
    checkpoint_name=CHECKPOINT_NAME,
    decode=DECODE,           # "greedy" o "beam"
)

df[out_col_name] = df["SMILES_out"]

# Si vols la llista:
smiles_out = df["SMILES_out"].tolist()

Passant MolForge per 2000 fingerprints vàlides (model carregat una sola vegada)...
[2000/2000]


## 3. Guardar l'output

In [10]:
# Creem el nou dataframe
df[out_col_name] = smiles_out
df.head(5)

,SMILES_input,fingerprints_input_ECFP4,SMILES_out,SMILES_output_ECFP4
id,,,,
1,C C 1 = C ( C = C C ( = C 1 ) C ( C ) ( C ) C ...,1 13 33 80 94 114 118 227 283 351 392 404 422 ...,CC1=C(C=CC(=C1)C(C)(C)C)OC[C@@H](C[NH2+]CCCOC)O,CC1=C(C=CC(=C1)C(C)(C)C)OC[C@@H](C[NH2+]CCCOC)O
2,C ( C ( F ) ( F ) F ) ( C ( F ) ( F ) [18F] ) F,1 114 136 218 322 456 545 561 676 1050 1059 10...,C(C(C(C(C(C(F)(F)F)F)(F)F)F)(F)F)(C(C(C(C(C(C(...,C(C(C(C(C(C(F)(F)F)F)(F)F)F)(F)F)(C(C(C(C(C(C(...
3,C C 1 = C C = C C = C 1 C ( C ( = O ) N C 2 C ...,14 80 106 145 196 237 241 325 330 383 387 391 ...,CC1=CC=CC=C1C(C(=O)NC2CCCC2)N(C3=CC=C(C=C3)C(=...,CC1=CC=CC=C1C(C(=O)NC2CCCC2)N(C3=CC=C(C=C3)C(=...
4,C C O C ( = O ) C 1 = C C = C ( C = C 1 ) N C ...,5 41 63 80 89 121 129 145 147 162 191 202 216 ...,CCOC(=O)C1=CC=C(C=C1)N2C(=O)C3C4CC(C3C2=O)C5C4...,CCOC(=O)C1=CC=C(C=C1)N2C(=O)C3C4CC(C3C2=O)C5C4...
5,C 1 [C@H] ( C N ( C 1 = O ) C C 2 = C C = C C ...,80 93 101 172 197 221 231 295 311 314 366 378 ...,C1C(CN(C1=O)CC2=CC=CC=C2Cl)C(=O)NCCC3=CC=CC=N3,C1C(CN(C1=O)CC2=CC=CC=C2Cl)C(=O)NCCC3=CC=CC=N3


In [11]:
df.to_csv(output_path)